# KADENCE demand forecaster -- NRP interactive training

Run this on the NRP JupyterHub after selecting a **low-cpu / low-ram** profile with an **A100 or L40** GPU.
The repo and the two `.npz` datasets must already be staged on the PVC at `/home/jovyan/kadence`.

Cells are small on purpose: run them top to bottom. The last cell runs the full sweep (long).

In [ ]:
# 1. GPU info -- confirm we actually landed on a powerful GPU.
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
else:
    print('WARNING: no CUDA visible -- training will fall back to CPU and be slow.')

In [ ]:
# 2. Put the repo root on sys.path so `import sim.forecast_models` works.
import os, sys
ROOT = '/home/jovyan/kadence'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
DATA = os.path.join(ROOT, 'data', 'gct_days.npz')
assert os.path.exists(DATA), f'dataset missing: {DATA} -- stage it onto the PVC first'
print('repo root on path; dataset found at', DATA)

In [ ]:
# 3. Import the training entry point.
from sim.forecast_models import train_eval, BACKBONES
print('backbones:', BACKBONES)

In [ ]:
# 4. Short all-backbones comparison on the Google trace (gct_days.npz).
# Modest epochs for interactive use; full-depth epochs live in the batch Job
# and in the final cell. max_tasks=None uses all 82,440 tasks (fine on a GPU).
results = {}
for bb in BACKBONES:
    print('--- training', bb, flush=True)
    results[bb] = train_eval(DATA, backbone=bb, epochs=40, max_tasks=None,
                             quantize=False, diagnose=True)
print('done')

In [ ]:
# 5. Small leaderboard: backbone, learned nodes, peak nodes, beats_peak.
# 'learned' is the model forecast at the reservation quantile; 'peak' is the
# simple peak-requests baseline. beats_peak is True only when the model uses no
# more nodes at no worse overload -- an honest bar.
print(f"{'backbone':<9} {'learned_nodes':>13} {'peak_nodes':>11} {'beats_peak':>11}")
for bb, r in results.items():
    if r.get('status') != 'ok':
        print(f"{bb:<9} {'(' + str(r.get('status')) + ')':>37}")
        continue
    ln = r['results']['learned_q']['nodes']
    pn = r['results']['peak']['nodes']
    print(f"{bb:<9} {ln:>13} {pn:>11} {str(r['beats_peak']):>11}")

## Full sweep (long-running)

The next cell runs the production sweep: every backbone on both datasets at
400 epochs with int8 quantization, writing `results/forecaster_sweep.json` on
the PVC. This is the same work the batch Job does. Start a heartbeat kernel /
keep the tab active so the NRP culler does not kill the server mid-run.

In [ ]:
# 6. Full sweep. Prefers the sweep harness if present, else calls train_eval.
import json, importlib.util
harness = os.path.join(ROOT, 'experiments', 'train_forecaster.py')
os.makedirs(os.path.join(ROOT, 'results'), exist_ok=True)
out_path = os.path.join(ROOT, 'results', 'forecaster_sweep.json')
datasets = [os.path.join(ROOT, 'data', 'gct_days.npz'),
            os.path.join(ROOT, 'data', 'alibaba_days.npz')]

if os.path.exists(harness):
    print('running sweep harness via runpy:', harness, flush=True)
    import runpy
    sys.argv = ['train_forecaster.py', '--epochs', '400', '--quantize',
                '--out', out_path]
    runpy.run_path(harness, run_name='__main__')
else:
    print('harness absent; calling train_eval directly', flush=True)
    runs = []
    for data in datasets:
        if not os.path.exists(data):
            print('skip missing', data); continue
        for bb in BACKBONES:
            print('=== train', bb, os.path.basename(data), flush=True)
            runs.append(train_eval(data, backbone=bb, epochs=400,
                                   max_tasks=None, quantize=True))
    json.dump({'runs': runs}, open(out_path, 'w'), indent=2)
    print('wrote', out_path)